# 02 Incident Partitioning

Freeze the incident-level Train / Validation / Final Holdout partitions for the balanced Cloudflare BGP incident roster. This notebook does not download MRT files, create 5-minute windows, generate Normal labels, extract features, train models, or run ablations.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from bgp_incident_detection.partitioning import (
    CLASS_LABELS,
    PARTITION_TARGETS,
    RANDOM_SEED,
    SPLIT_VERSION,
    audit_diversity,
    incident_uid,
    load_roster,
    manifest_rows,
    overlap_report,
    print_audit,
    split_incidents,
    validate_roster,
    validate_split,
    write_outputs,
)

ROSTER_PATH = ROOT / "cloudflare_incidents/balanced_incident_level_roster_473_each_leak_hijack_outage.csv"
OUTPUT_DIR = ROOT / "data/partitions"
ROSTER_PATH, OUTPUT_DIR, RANDOM_SEED, SPLIT_VERSION

## Load and Validate Authoritative Roster

In [ ]:
rows = load_roster(ROSTER_PATH)
roster_audit = validate_roster(rows)
roster_audit

## Perform Frozen Incident-Level Split

In [ ]:
assignments = split_incidents(rows, seed=RANDOM_SEED)
split_audit = validate_split(assignments, rows)
write_outputs(assignments, OUTPUT_DIR, seed=RANDOM_SEED)
split_audit

## Verify No Incident Leakage Across Partitions

In [ ]:
overlaps = overlap_report(assignments)
for label, count in overlaps.items():
    print(f"{label} = {count} incidents")
assert all(count == 0 for count in overlaps.values())

## Verify Every Incident Appears Exactly Once

In [ ]:
source_ids = {incident_uid(row) for row in rows}
assigned_ids = [row["incident_uid"] for row in assignments]
assert len(source_ids) == 1419
assert len(assigned_ids) == 1419
assert set(assigned_ids) == source_ids
assert len(assigned_ids) == len(set(assigned_ids))
print("Every incident appears in exactly one partition: PASS")

## Verify Final Counts

In [ ]:
from collections import Counter

partition_counts = Counter(row["partition"] for row in assignments)
class_partition_counts = Counter((row["partition"], row["category"]) for row in assignments)

print("Partition counts")
for partition in PARTITION_TARGETS:
    print(f"  {partition}: {partition_counts[partition]}")

print("\nPer-class counts")
for partition in PARTITION_TARGETS:
    for category in CLASS_LABELS:
        print(f"  {partition} {category}: {class_partition_counts[(partition, category)]}")

assert partition_counts["train"] == 993
assert partition_counts["validation"] == 213
assert partition_counts["holdout"] == 213
for partition, target in PARTITION_TARGETS.items():
    for category in CLASS_LABELS:
        assert class_partition_counts[(partition, category)] == target

## Secondary Diversity Audit

In [ ]:
for row in audit_diversity(assignments):
    print(row)

## PASS / REVIEW / FAIL Summary

In [ ]:
try:
    validate_roster(rows)
    validate_split(assignments, rows)
    if all(count == 0 for count in overlap_report(assignments).values()):
        print("PASS: frozen incident-level partitions are valid and non-overlapping")
    else:
        print("FAIL: partition overlap detected")
except Exception as exc:
    print(f"FAIL: {exc}")

print("REVIEW: diversity across year/date, duration, ASN/network, country, and prefix count is secondary and should be inspected before feature extraction.")
print("HOLDOUT RULE: final holdout is frozen and must not be used for threshold selection, feature selection, balancing decisions, hyperparameter tuning, or model comparison.")